# Data · NGCF 추천시스템 원본 빈칸

문제: **RecSys_GCF_practice.ipynb** / 정답: **RecSys_GCF_sol.ipynb**.
원본 전체 셀14·15의 빈칸8곳을5문제로 묶었습니다. 문제→빈칸→검증→접힌 해설·정답 순서입니다.

첫 준비 셀을 실행한 뒤 MISSING 또는 NotImplementedError 부분을 작성하세요. 함수 셀→검증 셀 순서로 실행합니다. CPU PyTorch만 필요하고 MovieLens 다운로드·torch-geometric·GPU·전체학습은 필요 없습니다. NGCF는 Neural Graph Collaborative Filtering(신경망 그래프 협업 필터링)입니다.

**원본 제공 코드 보정:** practice와 sol 모두 사용자 누적에서 `aggregated_messages.index_add(...)`의 반환값을 버립니다. index_add는 원본을 바꾸지 않으므로 메시지가 반영되지 않습니다. 이 자료의 제공 레이어는 `result=result.index_add(...)`로 보정했습니다(`index_add_`도 가능). 또한 원본 `zeros_like(node_features)`는 입력/출력 차원이 같을 때만 맞습니다. 제공 레이어는 W1 출력 shape를 사용합니다. 보정은 학습자 빈칸에 추가하지 않았으며 원본 파일도 수정하지 않았습니다.

이 노트북은 빈칸의 작은 그래프 연산을 검사합니다. 실제 추천 정확도·BPR 학습·Recall/NDCG 성능 평가는 포함하지 않습니다. SyntaxError가 나면 수정 후 함수 셀부터 다시 실행하세요.

In [ ]:
import torch
import torch.nn as nn
torch.set_num_threads(1)
MISSING=object()  # ???? 옆 MISSING을 정답으로 바꾸세요.

class MessageWeights(nn.Module):
    def __init__(self,dim,out_dim):
        super().__init__()
        self.W1=nn.Linear(dim,out_dim)
        self.W2=nn.Linear(dim,out_dim)

class SmallGraphLayer(nn.Module):
    # 이 문제의 의존성을 없애기 위한 제공 구현입니다.
    # 원본 index_add 결과 유실과 in/out 차원이 다를 때 누적 shape를 보정했습니다.
    def __init__(self,dim,out_dim):
        super().__init__()
        self.W1=nn.Linear(dim,out_dim)
        self.W2=nn.Linear(dim,out_dim)
        self.calls=[]
    def forward(self,edge_index,node_features,user_num,item_num):
        assert node_features.shape[0]==user_num+item_num
        self.calls.append((user_num,item_num))
        src,dst=edge_index
        degree=torch.bincount(edge_index.flatten(),minlength=node_features.shape[0]).float()
        norm=(degree[src]*degree[dst]).rsqrt().unsqueeze(1)
        user_message=(self.W1(node_features[dst])+self.W2(node_features[src]*node_features[dst]))*norm
        item_message=(self.W1(node_features[src])+self.W2(node_features[src]*node_features[dst]))*norm
        result=self.W1(node_features)
        result=result.index_add(0,src,user_message)
        result=result.index_add(0,dst,item_message)
        return torch.nn.functional.leaky_relu(result,negative_slope=.2)

def same(a,e):
    if isinstance(e,torch.Tensor):torch.testing.assert_close(a,e,rtol=1e-5,atol=1e-6)
    elif isinstance(e,(tuple,list)):
        assert len(a)==len(e)
        for x,y in zip(a,e):same(x,y)
    else:assert a==e,(a,e)

def snapshot(v):
    if isinstance(v,torch.Tensor):return v.detach().clone()
    if isinstance(v,(tuple,list)):return tuple(snapshot(x) for x in v)
    return v


## 01. 엣지별 degree 정규화

원본 **전체 셀14** (Markdown 포함,1부터), 빈칸1곳. `self`는 테스트에서 준비한 레이어/모델 객체입니다. 해당 이름을 원본과 같게 유지했습니다. 반환값: `norm`.

입력 예시:
```python
case=0
deg=torch.tensor([2.,1.,1.,2.,0.]) if case==0 else torch.tensor([3.,2.,2.,3.,0.])
src=torch.tensor([0,0,1])
dst=torch.tensor([2,3,3])
```

In [ ]:
import torch
import torch.nn as nn

def solve_01(deg, src, dst):
    norm = MISSING  # ???? 원본 빈칸
    return norm

In [ ]:
def check_01():
    expected=(torch.tensor([0.7071067690849304, 0.5, 0.7071067690849304],dtype=torch.float32), torch.tensor([0.40824827551841736, 0.3333333432674408, 0.40824827551841736],dtype=torch.float32),)
    for case in range(2):
        torch.manual_seed(4010+case)
        deg=torch.tensor([2.,1.,1.,2.,0.]) if case==0 else torch.tensor([3.,2.,2.,3.,0.])
        src=torch.tensor([0,0,1])
        dst=torch.tensor([2,3,3])
        actual=solve_01(deg, src, dst)
        same(snapshot(actual),expected[case])
    print('PASS · 01 · 두 그래프/입력 검증')
try:
    check_01()
except Exception as error:
    print('FAIL · 01 ·',type(error).__name__,str(error))


<details><summary>해설 · 공식 · 정답 보기</summary>

norm_e=1/sqrt(deg(user)×deg(item)). src,dst는(E,) 정수 인덱스이고 norm도(E,)입니다. 예: degree2와2가 연결되면 .5. 고립 노드 degree0도 있을 수 있지만 실제 엣지 양 끝의 degree는 양수입니다.

```python
def solve_01(deg, src, dst):
    norm = 1.0 / torch.sqrt(deg[src] * deg[dst])
    return norm
```

실습 빈칸 밖의 준비는 테스트가 제공합니다. 먼저 시도하고 해설을 확인하세요.
</details>

## 02. 아이템으로 전달하는 메시지

원본 **전체 셀14** (Markdown 포함,1부터), 빈칸2곳. `self`는 테스트에서 준비한 레이어/모델 객체입니다. 해당 이름을 원본과 같게 유지했습니다. 반환값: `edge_messages_for_dst`.

입력 예시:
```python
case=0
self=MessageWeights(3,case+2)
src_feat=torch.tensor([[1.,2.,-1.],[.5,-.2,1.],[2.,1.,.3]])
dst_feat=torch.tensor([[-1.,.3,2.],[1.,2.,.2],[.4,-1.,2.]])
norm=torch.tensor([.5,.25,.75])
```

In [ ]:
import torch
import torch.nn as nn

def solve_02(self, src_feat, dst_feat, norm):
    edge_messages_for_dst = MISSING  # ???? 원본 빈칸
    edge_messages_for_dst *= MISSING  # ???? 원본 빈칸
    return edge_messages_for_dst

In [ ]:
def check_02():
    expected=(torch.tensor([[-0.22436901926994324, -0.15974178910255432], [0.11664386838674545, -0.1164449006319046], [-0.04260020703077316, -0.0832137018442154]],dtype=torch.float32), torch.tensor([[-0.30143386125564575, 0.26692163944244385, -0.7254747152328491], [-0.3788350522518158, 0.32488954067230225, -0.015896707773208618], [-1.1190857887268066, 1.153823733329773, -1.093449592590332]],dtype=torch.float32),)
    for case in range(2):
        torch.manual_seed(4020+case)
        self=MessageWeights(3,case+2)
        src_feat=torch.tensor([[1.,2.,-1.],[.5,-.2,1.],[2.,1.,.3]])
        dst_feat=torch.tensor([[-1.,.3,2.],[1.,2.,.2],[.4,-1.,2.]])
        norm=torch.tensor([.5,.25,.75])
        actual=solve_02(self, src_feat, dst_feat, norm)
        same(snapshot(actual),expected[case])
        actual.square().mean().backward()
        assert all(p.grad is not None for p in self.parameters())
    print('PASS · 02 · 두 그래프/입력 검증')
try:
    check_02()
except Exception as error:
    print('FAIL · 02 ·',type(error).__name__,str(error))


<details><summary>해설 · 공식 · 정답 보기</summary>

item←user 메시지=norm×(W1(h_user)+W2(h_user⊙h_item)). ⊙는 원소별 곱이며 @가 아닙니다. (E,Din)→Linear→(E,Dout),norm(E,)→unsqueeze(1)=(E,1). 예: E3,Dout2이면 각 행을 자기 norm으로 곱합니다.

```python
def solve_02(self, src_feat, dst_feat, norm):
    edge_messages_for_dst = self.W1(src_feat) + self.W2(src_feat * dst_feat)
    edge_messages_for_dst *= norm.unsqueeze(1)
    return edge_messages_for_dst
```

실습 빈칸 밖의 준비는 테스트가 제공합니다. 먼저 시도하고 해설을 확인하세요.
</details>

## 03. 아이템 메시지 누적과 자기 정보

원본 **전체 셀14** (Markdown 포함,1부터), 빈칸1곳. `self`는 테스트에서 준비한 레이어/모델 객체입니다. 해당 이름을 원본과 같게 유지했습니다. 반환값: `aggregated_messages`.

입력 예시:
```python
case=0
self=MessageWeights(3,3)
node_features=torch.randn(5,3)
user_num=2
dst=torch.tensor([2,3,3,4])
edge_messages_for_dst=torch.arange(12).float().reshape(4,3)*(case+1)
aggregated_messages=torch.zeros_like(node_features)
aggregated_messages[:2]=torch.tensor([[1.,2.,3.],[4.,5.,6.]])
```

In [ ]:
import torch
import torch.nn as nn

def solve_03(self, aggregated_messages, dst, edge_messages_for_dst, node_features, user_num):
    raise NotImplementedError('???? dst별 누적과 item 자기 메시지 두 줄을 구현하세요')
    return aggregated_messages

In [ ]:
def check_03():
    expected=(torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [-0.2615119516849518, 1.8615751266479492, 1.4051010608673096], [10.002596855163574, 10.926013946533203, 14.373664855957031], [8.92473030090332, 10.555950164794922, 11.401897430419922]],dtype=torch.float32), torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [0.8175948262214661, 2.147474765777588, 3.862257719039917], [18.3826961517334, 21.753664016723633, 25.512304306030273], [19.09897804260254, 19.863601684570312, 22.762187957763672]],dtype=torch.float32),)
    for case in range(2):
        torch.manual_seed(4030+case)
        self=MessageWeights(3,3)
        node_features=torch.randn(5,3)
        user_num=2
        dst=torch.tensor([2,3,3,4])
        edge_messages_for_dst=torch.arange(12).float().reshape(4,3)*(case+1)
        aggregated_messages=torch.zeros_like(node_features)
        aggregated_messages[:2]=torch.tensor([[1.,2.,3.],[4.,5.,6.]])
        actual=solve_03(self, aggregated_messages, dst, edge_messages_for_dst, node_features, user_num)
        same(snapshot(actual),expected[case])
    print('PASS · 03 · 두 그래프/입력 검증')
try:
    check_03()
except Exception as error:
    print('FAIL · 03 ·',type(error).__name__,str(error))


<details><summary>해설 · 공식 · 정답 보기</summary>

index_add_(0,dst,message)는 같은 dst로 온 메시지를 더합니다. 예: dst=[3,3],메시지[[1,2],[4,5]]→3번 행에[5,7]. 이어 item 부분에 W1(h_item)을 한 번 더합니다. user 행은 그대로 보존합니다. input_dim=output_dim인 원본 조건입니다.

```python
def solve_03(self, aggregated_messages, dst, edge_messages_for_dst, node_features, user_num):
    aggregated_messages.index_add_(0, dst, edge_messages_for_dst)
    aggregated_messages[user_num:] += self.W1(node_features[user_num:])
    return aggregated_messages
```

실습 빈칸 밖의 준비는 테스트가 제공합니다. 먼저 시도하고 해설을 확인하세요.
</details>

## 04. NGCF 레이어 순서대로 호출

원본 **전체 셀15** (Markdown 포함,1부터), 빈칸1곳. `self`는 테스트에서 준비한 레이어/모델 객체입니다. 해당 이름을 원본과 같게 유지했습니다. 반환값: `layer_outputs`.

입력 예시:
```python
case=0
self=nn.Module()
self.num_users=2
self.num_items=3
self.node_embeddings=nn.Embedding(5,3)
self.layers=nn.ModuleList([SmallGraphLayer(3,case+2),SmallGraphLayer(case+2,4)])
edge_index=torch.tensor([[0,0,1,1],[2,3,3,4]])
```

In [ ]:
import torch
import torch.nn as nn

def solve_04(self, edge_index):
    node_features=self.node_embeddings.weight
    layer_outputs=[node_features]
    for layer in self.layers:
        node_features=MISSING  # ???? 원본 빈칸
        layer_outputs.append(node_features)
    return layer_outputs

In [ ]:
def check_04():
    expected=((torch.tensor([[0.7347200512886047, -0.0035741357132792473, -1.3707547187805176], [-0.6705525517463684, 0.9572908878326416, 0.1537402719259262], [0.7229401469230652, -0.15902802348136902, 1.6809731721878052], [-0.08319038897752762, -1.4362480640411377, 1.4042123556137085], [-0.5396146178245544, 0.6414323449134827, 2.864304542541504]],dtype=torch.float32), torch.tensor([[-0.3937540650367737, -0.09382092952728271], [-0.4355158805847168, -0.33926302194595337], [-0.3506198525428772, -0.008470329456031322], [-0.17040704190731049, -0.345077246427536], [-0.46336206793785095, -0.11761905997991562]],dtype=torch.float32), torch.tensor([[-0.16038770973682404, -0.3614646792411804, -0.2188386470079422, 0.5044883489608765], [-0.18828420341014862, -0.3684808909893036, -0.1888011246919632, 0.5541898012161255], [-0.12325545400381088, -0.2380279153585434, -0.1567768156528473, 0.24978163838386536], [-0.18963897228240967, -0.34198877215385437, -0.16746117174625397, 0.5120624303817749], [-0.1429954320192337, -0.2382320910692215, -0.1286075860261917, 0.2688051760196686]],dtype=torch.float32),), (torch.tensor([[-0.511130154132843, -0.7908685207366943, 1.3973584175109863], [-0.8610146641731262, -0.9916316866874695, 0.19207951426506042], [-0.7126874923706055, 0.1102362722158432, 0.031870149075984955], [0.8480901718139648, 0.4222066104412079, -0.878078818321228], [-0.9517367482185364, 0.4531002640724182, 0.15404130518436432]],dtype=torch.float32), torch.tensor([[0.6499885320663452, 0.07561972737312317, 0.46906113624572754], [0.2930891215801239, 0.28131231665611267, 0.03373432159423828], [0.2595588266849518, -0.02382456138730049, 0.34760355949401855], [0.7979823350906372, 0.16215133666992188, -0.0598684661090374], [0.20470324158668518, 0.15119555592536926, 0.35427436232566833]],dtype=torch.float32), torch.tensor([[-0.02676813304424286, -0.1328844130039215, -0.1632411628961563, 0.38287249207496643], [0.1726914793252945, -0.16747736930847168, -0.11718855053186417, 0.1650690883398056], [-0.024304959923028946, -0.048348575830459595, -0.11172761768102646, 0.6794419884681702], [-0.009482534602284431, -0.1639440953731537, -0.16104377806186676, -0.00032872558222152293], [0.09773500263690948, -0.08738858997821808, -0.09106775373220444, 0.4486857056617737]],dtype=torch.float32),),)
    for case in range(2):
        torch.manual_seed(4040+case)
        self=nn.Module()
        self.num_users=2
        self.num_items=3
        self.node_embeddings=nn.Embedding(5,3)
        self.layers=nn.ModuleList([SmallGraphLayer(3,case+2),SmallGraphLayer(case+2,4)])
        edge_index=torch.tensor([[0,0,1,1],[2,3,3,4]])
        actual=solve_04(self, edge_index)
        same(snapshot(actual),expected[case])
        assert all(layer.calls==[(self.num_users,self.num_items)] for layer in self.layers)
    print('PASS · 04 · 두 그래프/입력 검증')
try:
    check_04()
except Exception as error:
    print('FAIL · 04 ·',type(error).__name__,str(error))


<details><summary>해설 · 공식 · 정답 보기</summary>

처음 임베딩 H0를 보관하고 H1=layer1(edge,H0,U,I),H2=layer2(edge,H1,U,I)를 계산합니다. 다음 레이어에는 갱신한 특징을 넣습니다. 테스트의 제공 레이어는 누적 버그를 보정했고 서로 다른 출력 차원도 지원합니다.

```python
def solve_04(self, edge_index):
    node_features=self.node_embeddings.weight
    layer_outputs=[node_features]
    for layer in self.layers:
        node_features=layer(edge_index,node_features,self.num_users,self.num_items)
        layer_outputs.append(node_features)
    return layer_outputs
```

실습 빈칸 밖의 준비는 테스트가 제공합니다. 먼저 시도하고 해설을 확인하세요.
</details>

## 05. 레이어 특징 결합과 user/item 분리

원본 **전체 셀15** (Markdown 포함,1부터), 빈칸3곳. `self`는 테스트에서 준비한 레이어/모델 객체입니다. 해당 이름을 원본과 같게 유지했습니다. 반환값: `(user_features,item_features)`.

입력 예시:
```python
case=0
self=nn.Module()
self.num_users=2+case
self.num_items=3
total=self.num_users+self.num_items
layer_outputs=[torch.arange(total*d).float().reshape(total,d)+100*i for i,d in enumerate([2,3,4])]
```

In [ ]:
import torch
import torch.nn as nn

def solve_05(self, layer_outputs):
    final_features=MISSING  # ???? 원본 빈칸
    user_features=MISSING  # ???? 원본 빈칸
    item_features=MISSING  # ???? 원본 빈칸
    return (user_features,item_features)

In [ ]:
def check_05():
    expected=((torch.tensor([[0.0, 1.0, 100.0, 101.0, 102.0, 200.0, 201.0, 202.0, 203.0], [2.0, 3.0, 103.0, 104.0, 105.0, 204.0, 205.0, 206.0, 207.0]],dtype=torch.float32), torch.tensor([[4.0, 5.0, 106.0, 107.0, 108.0, 208.0, 209.0, 210.0, 211.0], [6.0, 7.0, 109.0, 110.0, 111.0, 212.0, 213.0, 214.0, 215.0], [8.0, 9.0, 112.0, 113.0, 114.0, 216.0, 217.0, 218.0, 219.0]],dtype=torch.float32),), (torch.tensor([[0.0, 1.0, 100.0, 101.0, 102.0, 200.0, 201.0, 202.0, 203.0], [2.0, 3.0, 103.0, 104.0, 105.0, 204.0, 205.0, 206.0, 207.0], [4.0, 5.0, 106.0, 107.0, 108.0, 208.0, 209.0, 210.0, 211.0]],dtype=torch.float32), torch.tensor([[6.0, 7.0, 109.0, 110.0, 111.0, 212.0, 213.0, 214.0, 215.0], [8.0, 9.0, 112.0, 113.0, 114.0, 216.0, 217.0, 218.0, 219.0], [10.0, 11.0, 115.0, 116.0, 117.0, 220.0, 221.0, 222.0, 223.0]],dtype=torch.float32),),)
    for case in range(2):
        torch.manual_seed(4050+case)
        self=nn.Module()
        self.num_users=2+case
        self.num_items=3
        total=self.num_users+self.num_items
        layer_outputs=[torch.arange(total*d).float().reshape(total,d)+100*i for i,d in enumerate([2,3,4])]
        actual=solve_05(self, layer_outputs)
        same(snapshot(actual),expected[case])
    print('PASS · 05 · 두 그래프/입력 검증')
try:
    check_05()
except Exception as error:
    print('FAIL · 05 ·',type(error).__name__,str(error))


<details><summary>해설 · 공식 · 정답 보기</summary>

노드 수는 그대로,특징 축을 연결합니다. 예: (5,2)+(5,3)+(5,4)→concat(5,9). U=2이면 users(2,9),items(3,9). node 번호는 user가 앞, item이 뒤입니다. 더하기나 노드축 cat은 다릅니다.

```python
def solve_05(self, layer_outputs):
    final_features=torch.concat(layer_outputs,dim=-1)
    user_features=final_features[:self.num_users]
    item_features=final_features[self.num_users:]
    return (user_features,item_features)
```

실습 빈칸 밖의 준비는 테스트가 제공합니다. 먼저 시도하고 해설을 확인하세요.
</details>